# SkinTell — 03 Modeling
Train a skin condition severity classifier using transfer learning with MobileNetV2.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

CLEAN_DIR  = '../data/processed'
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 4
CLASS_LABELS = ['clear', 'mild', 'moderate', 'severe']

print(f'TensorFlow: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

## 1. Data Generators

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255, validation_split=0.2,
    rotation_range=20, horizontal_flip=True,
    brightness_range=[0.8, 1.2], zoom_range=0.15,
    width_shift_range=0.1, height_shift_range=0.1
)
val_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train_gen = train_datagen.flow_from_directory(
    CLEAN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='training', seed=42
)
val_gen = val_datagen.flow_from_directory(
    CLEAN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='validation', seed=42
)

class_weights = compute_class_weight('balanced', classes=np.unique(train_gen.classes), y=train_gen.classes)
class_weight_dict = dict(enumerate(class_weights))
print('Class weights:', {CLASS_LABELS[k]: round(v, 3) for k, v in class_weight_dict.items()})

## 2. Build Model — MobileNetV2 Transfer Learning

In [ ]:
base_model = MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
base_model.trainable = False  # Freeze base initially

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(NUM_CLASSES, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

## 3. Phase 1 — Train Top Layers (base frozen)

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ModelCheckpoint('../models/best_model.keras', monitor='val_accuracy', save_best_only=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)
]

history1 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=15,
    class_weight=class_weight_dict,
    callbacks=callbacks
)

## 4. Phase 2 — Fine-tune Top Layers of Base Model

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history2 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=30,
    class_weight=class_weight_dict,
    callbacks=callbacks
)

## 5. Plot Training History

In [ ]:
def plot_history(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(history.history['accuracy'], label='Train', color='#c4a7e7')
    axes[0].plot(history.history['val_accuracy'], label='Validation', color='#9ccfd8')
    axes[0].set_title(f'Accuracy — {title}', fontweight='bold')
    axes[0].set_xlabel('Epoch')
    axes[0].legend()
    axes[1].plot(history.history['loss'], label='Train', color='#e07b7b')
    axes[1].plot(history.history['val_loss'], label='Validation', color='#f6c177')
    axes[1].set_title(f'Loss — {title}', fontweight='bold')
    axes[1].set_xlabel('Epoch')
    axes[1].legend()
    plt.tight_layout()
    plt.savefig(f'../plots/history_{title.lower().replace(" ", "_")}.png', dpi=150)
    plt.show()

plot_history(history1, 'Phase 1')
plot_history(history2, 'Phase 2 Fine-tuning')